# Create Scenes

(advanced:create-scenes)=

This tutorial demonstrates basic usage around writing and reading [Ngff Scenes](https://ngff.openmicroscopy.org/specifications/dev/index.html#scene-metadata) using the {py:class}`ome_zarr.classes.scene.NgffScene` class.

In [6]:
from ome_zarr import NgffImage, NgffMultiscales, NgffScene
from skimage import data
import napari

We create some sample data, whioch we will store as a tiled layout in a scene zarr group:

In [2]:
example_image = data.human_mitosis()
example_image.shape

(512, 512)

First, we cut the image into four tiles and convert them into instances of {py:class}`ome_zarr.classes.image.NgffMultiscales`:

In [3]:
img1 = NgffImage(data=example_image[:256, :256], axes=["y", "x"], name="img1")
img2 = NgffImage(data=example_image[256:, :256], axes=["y", "x"], name="img2")
img3 = NgffImage(data=example_image[:256, 256:], axes=["y", "x"], name="img3")
img4 = NgffImage(data=example_image[256:, 256:], axes=["y", "x"], name="img4")

img1_ms = NgffMultiscales(img1)
img2_ms = NgffMultiscales(img2)
img3_ms = NgffMultiscales(img3)
img4_ms = NgffMultiscales(img4)

name='physical' axes=(Axis(name='y', type='space', discrete=None, unit=None, longName=None), Axis(name='x', type='space', discrete=None, unit=None, longName=None))
name='physical' axes=(Axis(name='y', type='space', discrete=None, unit=None, longName=None), Axis(name='x', type='space', discrete=None, unit=None, longName=None))
name='physical' axes=(Axis(name='y', type='space', discrete=None, unit=None, longName=None), Axis(name='x', type='space', discrete=None, unit=None, longName=None))
name='physical' axes=(Axis(name='y', type='space', discrete=None, unit=None, longName=None), Axis(name='x', type='space', discrete=None, unit=None, longName=None))


Next, we need to define a coordinate system into which all images are projected.
This is defined in accordance with the NGFF [coordinate systems specification](https://ngff.openmicroscopy.org/specifications/dev/index.html#coordinatesystems-metadata).

In [17]:
coordinate_system = {
    "name": "world",
    "axes": [
        {"name": "y", "type": "space"},
        {"name": "x", "type": "space"},
    ],
}

We then define transdlations that move each tile into the appropriate position in the world coordinate system.
In thisexamples, these are simple translations in the y and x dimensions:

In [18]:
coordinate_transformations = [
    {
        "type": "translation",
        "translation": [0, 0],
        "input": {"path": "img1", "name": "physical"},
        "output": {"name": "world"},
    },
    {
        "type": "translation",
        "translation": [256, 0],
        "input": {"path": "img2", "name": "physical"},
        "output": {"name": "world"},
    },
    {
        "type": "translation",
        "translation": [0, 256],
        "input": {"path": "img3", "name": "physical"},
        "output": {"name": "world"},
    },
    {
        "type": "translation",
        "translation": [256, 256],
        "input": {"path": "img4", "name": "physical"},
        "output": {"name": "world"},
    },
]

We can then create and write a scene like this:

In [19]:
scene = NgffScene(
    images=[img1_ms, img2_ms, img3_ms, img4_ms],
    coordinate_systems=[coordinate_system],
    coordinate_transformations=coordinate_transformations
)

scene.to_ome_zarr("test_example_scene.zarr", overwrite=True)

## Optional: View with napari

In [ ]:
viewer = napari.Viewer()

In [13]:
viewer.layers.clear()
viewer.open("test_example_scene.zarr", plugin="napari-ome-zarr")
viewer.add_image(example_image, name="original image", blending="additive", colormap="orange")

Root group {'ome': {'scene': {'coordinateTransformations': [{'type': 'translation', 'input': {'name': 'physical', 'path': 'img1'}, 'output': {'name': 'world'}, 'translation': [0.0, 0.0]}, {'type': 'translation', 'input': {'name': 'physical', 'path': 'img2'}, 'output': {'name': 'world'}, 'translation': [256.0, 0.0]}, {'type': 'translation', 'input': {'name': 'physical', 'path': 'img3'}, 'output': {'name': 'world'}, 'translation': [0.0, 256.0]}, {'type': 'translation', 'input': {'name': 'physical', 'path': 'img4'}, 'output': {'name': 'world'}, 'translation': [256.0, 256.0]}], 'coordinateSystems': [{'name': 'world', 'axes': [{'name': 'y', 'type': 'space'}, {'name': 'x', 'type': 'space'}]}]}, 'version': '0.6'}}
Scene.matches {'scene': {'coordinateTransformations': [{'type': 'translation', 'input': {'name': 'physical', 'path': 'img1'}, 'output': {'name': 'world'}, 'translation': [0.0, 0.0]}, {'type': 'translation', 'input': {'name': 'physical', 'path': 'img2'}, 'output': {'name': 'world'}, 

<Image layer 'original image' at 0x153c0adbb00>